# Module 5 · Agent-to-Agent (A2A) Protocols & Multi-Agent Systems

**Track:** Main Track — GenAI / Agentic Systems  
**Toolchain:** Pydantic · asyncio · JSON-RPC · Agent Protocol  
**Prerequisites:** NB24 (Agentic Orchestration & Memory)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---

## Beyond Single Agents

NB24 taught you how to build a single agent with tools, memory, and a state machine. But production AI systems increasingly require **teams** of specialists.

| Scenario | Single Agent Problem | Multi-Agent Solution |
|----------|---------------------|---------------------|
| Software dev | One agent writes AND reviews its code (conflict of interest) | Coder writes → Reviewer critiques → Tester validates |
| Context limits | One agent gets confused with 50 tools | 5 tools × 10 specialized agents |
| Cross-org AI | Your AI needs to use another company's AI | Standardized A2A protocols |
| Reliability | One agent failure = total failure | Graceful delegation, retries, fallbacks |

### Multi-Agent Topologies

```
1. Hierarchical (Manager-Worker)     2. Sequential (Pipeline)        3. Peer Chat (War Room)
   [Manager]                            [Research] → [Code] → [QA]     [Coder] ↔ [Critic]
    ↙  ↓  ↘                                                              ↕
[R] [E] [QA]                                                          [User]
```

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors give one LLM a massive prompt and pray. Seniors decompose problems into micro-agents with strict interfaces — each agent has a defined role, limited tools, and communicates via typed Pydantic schemas. This mirrors microservice architecture: small, focused services with clear APIs.

**Mental Model:** Multi-agent = corporate org chart. The CEO (manager) doesn't write code. They decompose the objective into tasks and delegate to department heads (specialist agents), who report results back in a structured format.

**Common Junior Pitfall:** Letting agents communicate in unstructured natural language. In production, agents MUST exchange typed Pydantic schemas — otherwise you get unparseable responses, broken handoffs, and cascading failures.

---

## 📑 Table of Contents

* [Beyond Single Agents](#beyond-single-agents)
  * [Multi-Agent Topologies](#multi-agent-topologies)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Building a Multi-Agent Team from Scratch](#1-building-a-multi-agent-team-from-scratch)
  * [1.1 The Agent Base Class](#11-the-agent-base-class)
  * [1.3 The Manager Agent (Orchestrator)](#13-the-manager-agent-orchestrator)
* [2. Shared Memory & Agent Communication](#2-shared-memory-agent-communication)
  * [2.1 The Agent Blackboard Pattern](#21-the-agent-blackboard-pattern)
  * [2.2 Structured Message Passing](#22-structured-message-passing)
* [3. Failure Handling & Circuit Breakers](#3-failure-handling-circuit-breakers)
* [4. A2A Protocols: The HTTP of Agents](#4-a2a-protocols-the-http-of-agents)
  * [4.1 Google's A2A Specification](#41-googles-a2a-specification)
  * [4.2 Cross-Organization Agent Communication](#42-cross-organization-agent-communication)
* [5. Framework Comparison (2026)](#5-framework-comparison-2026)
  * [⚖️ The Fundamental Trade-off](#the-fundamental-trade-off)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Why is the Manager-Worker pattern more reliable than peer-to-peer agents?](#q1-why-is-the-manager-worker-pattern-more-reliable-than-peer-to-peer-agents)
  * [Q2: Why must agents communicate via Pydantic schemas, not free-form text?](#q2-why-must-agents-communicate-via-pydantic-schemas-not-free-form-text)
  * [Q3: An agent fails 5 times in a row. What should happen?](#q3-an-agent-fails-5-times-in-a-row-what-should-happen)
  * [Q4: What is an Agent Card and why is it published at `.well-known/agent.json`?](#q4-what-is-an-agent-card-and-why-is-it-published-at-well-knownagentjson)
  * [Q5: When would you use CrewAI vs LangGraph for a multi-agent system?](#q5-when-would-you-use-crewai-vs-langgraph-for-a-multi-agent-system)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)


In [ ]:
# Cell 1 — Install
!pip install -q pydantic

## 1. Building a Multi-Agent Team from Scratch

### 1.1 The Agent Base Class

Every agent in a multi-agent system needs:
- **Identity:** A unique name and role
- **Tools:** A limited set of capabilities
- **Memory access:** Read/write to shared state
- **Structured I/O:** Typed input/output contracts

In [ ]:
# Cell 2 — Multi-agent system: Base class + Specialist agents
import time, json, random, uuid
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
from enum import Enum

class TaskStatus(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    NEEDS_REVIEW = "needs_review"

@dataclass
class TaskResult:
    """Typed output from any agent — enforces structured communication."""
    agent_name: str
    task_id: str
    status: TaskStatus
    output: str
    artifacts: Dict[str, Any] = field(default_factory=dict)
    error: Optional[str] = None
    duration_ms: float = 0.0
    cost_usd: float = 0.0

class BaseAgent:
    """Base class for all agents in the multi-agent system.
    
    Every agent:
    1. Has a name, role, and limited tool set
    2. Reads/writes to a shared Blackboard (Section 2)
    3. Returns structured TaskResult objects
    4. Tracks cost and has a budget limit
    """
    def __init__(self, name: str, role: str, tools: list, max_cost: float = 1.0):
        self.name = name
        self.role = role
        self.tools = tools
        self.max_cost = max_cost
        self.total_cost = 0.0
        self.execution_log = []
    
    def execute(self, task: str, context: dict = None) -> TaskResult:
        """Execute a task. Subclasses override this."""
        raise NotImplementedError
    
    def _track_cost(self, cost: float):
        self.total_cost += cost
        if self.total_cost > self.max_cost:
            raise RuntimeError(f"[{self.name}] Budget exceeded: ${self.total_cost:.3f} > ${self.max_cost}")


# === SPECIALIST AGENTS ===

class ResearchAgent(BaseAgent):
    """Gathers information from external sources."""
    def __init__(self):
        super().__init__("Alice", "Researcher", ["web_search", "paper_db", "competitor_analysis"])
    
    def execute(self, task: str, context: dict = None) -> TaskResult:
        start = time.time()
        self._track_cost(0.01)
        # Simulate research with realistic output
        findings = {
            "sources_checked": 12,
            "relevant_results": 5,
            "key_findings": [
                "Competitor A launched similar feature in Q3 with 15% adoption",
                "Market size estimated at $2.1B by 2027 (Gartner)",
                "Top user pain point: latency > 200ms causes 40% abandonment",
            ],
            "recommendation": "Proceed — market opportunity validated, focus on sub-100ms latency"
        }
        return TaskResult(
            agent_name=self.name, task_id=str(uuid.uuid4())[:8],
            status=TaskStatus.COMPLETED, output=json.dumps(findings, indent=2),
            artifacts={"research_data": findings},
            duration_ms=(time.time()-start)*1000, cost_usd=0.01
        )

class CodeAgent(BaseAgent):
    """Writes and implements code based on specifications."""
    def __init__(self):
        super().__init__("Bob", "Engineer", ["python", "sql", "github"])
    
    def execute(self, task: str, context: dict = None) -> TaskResult:
        start = time.time()
        self._track_cost(0.02)
        code_output = """
class ChatFeature:
    def __init__(self, model="gpt-4o-mini", max_latency_ms=100):
        self.model = model
        self.max_latency = max_latency_ms
    
    async def respond(self, user_msg: str) -> str:
        # Stream response for perceived speed
        response = await litellm.acompletion(
            model=self.model, messages=[{"role": "user", "content": user_msg}],
            stream=True, timeout=self.max_latency/1000
        )
        return response
"""
        return TaskResult(
            agent_name=self.name, task_id=str(uuid.uuid4())[:8],
            status=TaskStatus.NEEDS_REVIEW, output=code_output.strip(),
            artifacts={"code": code_output.strip(), "language": "python", "lines": 12},
            duration_ms=(time.time()-start)*1000, cost_usd=0.02
        )

class ReviewAgent(BaseAgent):
    """Reviews code and provides feedback."""
    def __init__(self):
        super().__init__("Charlie", "Code Reviewer", ["linter", "security_scan", "pytest"])
    
    def execute(self, task: str, context: dict = None) -> TaskResult:
        start = time.time()
        self._track_cost(0.015)
        code = context.get("code", "") if context else ""
        issues_found = random.choice([0, 1, 2])  # Simulate review findings
        review = {
            "verdict": "APPROVED" if issues_found == 0 else "CHANGES_REQUESTED",
            "issues_found": issues_found,
            "feedback": [
                "Add error handling for timeout exceptions",
                "Consider adding rate limiting middleware",
            ][:issues_found],
            "security_scan": "PASSED",
            "test_coverage": "78%",
        }
        status = TaskStatus.COMPLETED if issues_found == 0 else TaskStatus.NEEDS_REVIEW
        return TaskResult(
            agent_name=self.name, task_id=str(uuid.uuid4())[:8],
            status=status, output=json.dumps(review, indent=2),
            artifacts={"review": review},
            duration_ms=(time.time()-start)*1000, cost_usd=0.015
        )

print("✅ Agent classes defined: ResearchAgent, CodeAgent, ReviewAgent")
print(f"   Each agent has: typed TaskResult output, cost tracking, budget limits")

### 1.3 The Manager Agent (Orchestrator)

The Manager doesn't do work — it **decomposes, delegates, and synthesizes**. This is the hierarchical topology in action.

In [ ]:
# Cell 3 — Manager Agent with delegation, retry, and synthesis

class ManagerAgent:
    """Orchestrates a team of specialist agents.
    
    The Manager:
    1. Decomposes the objective into sub-tasks
    2. Delegates each sub-task to the right specialist
    3. Handles failures with retries and fallbacks
    4. Synthesizes results into a final deliverable
    5. Enforces total budget across all agents
    """
    def __init__(self, team: List[BaseAgent], total_budget: float = 0.50):
        self.team = {agent.role: agent for agent in team}
        self.total_budget = total_budget
        self.spent = 0.0
        self.execution_trace = []  # Full audit trail
    
    def _delegate(self, role: str, task: str, context: dict = None, max_retries: int = 2) -> TaskResult:
        """Delegate a task to a specialist with retry logic."""
        agent = self.team.get(role)
        if not agent:
            return TaskResult(agent_name="Manager", task_id="err", status=TaskStatus.FAILED,
                            output="", error=f"No agent with role: {role}")
        
        for attempt in range(max_retries):
            try:
                result = agent.execute(task, context)
                self.spent += result.cost_usd
                self.execution_trace.append({
                    "agent": agent.name, "role": role, "task": task[:50],
                    "status": result.status.value, "attempt": attempt + 1,
                    "cost": result.cost_usd
                })
                
                if self.spent > self.total_budget:
                    print(f"  ⚠️  Budget warning: ${self.spent:.3f} / ${self.total_budget:.2f}")
                
                if result.status != TaskStatus.FAILED:
                    return result
                print(f"  🔄 [{agent.name}] attempt {attempt+1} failed, retrying...")
            except RuntimeError as e:
                print(f"  💥 [{agent.name}] Error: {e}")
                return TaskResult(agent_name=agent.name, task_id="err",
                                status=TaskStatus.FAILED, output="", error=str(e))
        
        return TaskResult(agent_name=agent.name, task_id="err",
                        status=TaskStatus.FAILED, output="", error="Max retries exceeded")
    
    def run(self, objective: str):
        """Execute the full multi-agent workflow."""
        print(f"👔 [Manager] Objective: \"{objective}\"")
        print(f"👔 [Manager] Team: {[f\"{a.name} ({a.role})\" for a in self.team.values()]}")
        print(f"👔 [Manager] Budget: ${self.total_budget:.2f}")
        sep = "=" * 60
        print(sep)
        
        # Step 1: Research
        print(f"\n📋 Phase 1: Research")
        research = self._delegate("Researcher", "Analyze market for AI chat feature")
        print(f"  ✅ [{research.agent_name}] {research.status.value} ({research.duration_ms:.0f}ms, ${research.cost_usd:.3f})")
        
        if research.status == TaskStatus.FAILED:
            print(f"  ❌ Research failed: {research.error}. Aborting.")
            return research
        
        # Step 2: Code (depends on research)
        print(f"\n📋 Phase 2: Implementation")
        code_result = self._delegate("Engineer", "Implement chat feature", 
                                    context=research.artifacts.get("research_data", {}))
        print(f"  ✅ [{code_result.agent_name}] {code_result.status.value} ({code_result.duration_ms:.0f}ms, ${code_result.cost_usd:.3f})")
        
        # Step 3: Review (depends on code)
        print(f"\n📋 Phase 3: Code Review")
        review = self._delegate("Code Reviewer", "Review the implementation",
                               context=code_result.artifacts)
        print(f"  ✅ [{review.agent_name}] {review.status.value} ({review.duration_ms:.0f}ms, ${review.cost_usd:.3f})")
        
        # If review requests changes, send back to engineer
        review_data = review.artifacts.get("review", {})
        if review_data.get("verdict") == "CHANGES_REQUESTED":
            print(f"\n📋 Phase 3b: Revision (reviewer requested changes)")
            print(f"  Feedback: {review_data.get(\"feedback\", [])}")
            code_result = self._delegate("Engineer", "Fix issues from code review",
                                        context={**code_result.artifacts, "feedback": review_data})
            print(f"  ✅ [{code_result.agent_name}] revised ({code_result.duration_ms:.0f}ms)")
        
        # Synthesis
        sep2 = "=" * 60
        print(f"\n{sep2}")
        print("👔 [Manager] All phases complete.")
        print(f"   Total cost: ${self.spent:.3f} / ${self.total_budget:.2f} budget")
        trace_count = len(self.execution_trace)
        print(f"   Agents used: {trace_count} delegations")
        print(f"\n📊 Execution Trace:")
        for i, step in enumerate(self.execution_trace):
            print(f"   {i+1}. [{step[\"agent\"]:8s}] {step[\"status\"]:14s} | {step[\"task\"]} (${step[\"cost\"]:0.3f})")
        
        return code_result

# === RUN THE MULTI-AGENT TEAM ===
random.seed(42)  # For reproducible demo
team = [ResearchAgent(), CodeAgent(), ReviewAgent()]
manager = ManagerAgent(team, total_budget=0.50)
result = manager.run("Build an AI-powered chat feature for our customer support app")

---

## 2. Shared Memory & Agent Communication

### 2.1 The Agent Blackboard Pattern

The **Blackboard** is a shared memory space that all agents can read from and write to. It solves the problem of passing context between agents without tight coupling.

```
┌─────────────────────────────────────┐
│           BLACKBOARD                │
│                                     │
│  research_findings: {...}           │
│  code_artifact: "class Chat..."     │
│  review_verdict: "APPROVED"         │
│  budget_remaining: $0.42            │
│  conversation_history: [...]        │
└──────┬────────┬────────┬────────────┘
       │        │        │
   [Research] [Code]  [Review]   ← All agents R/W
```

### 2.2 Structured Message Passing

In [ ]:
# Cell 4 — Blackboard shared memory + structured message passing
from pydantic import BaseModel, Field
from typing import Literal
import threading

class AgentMessage(BaseModel):
    """Typed message for inter-agent communication.
    Agents NEVER communicate in free-form text — always structured schemas.
    """
    sender: str
    recipient: str
    msg_type: Literal["task_request", "task_result", "info_query", "error"]
    conversation_id: str
    payload: dict
    priority: Literal["low", "medium", "high", "critical"] = "medium"

class Blackboard:
    """Thread-safe shared memory for multi-agent systems.
    
    In production: Redis/Valkey (NB24 Tier 1 memory)
    Here: Python dict with threading lock for safety.
    """
    def __init__(self):
        self._store = {}
        self._message_queue = []  # Inter-agent message bus
        self._lock = threading.Lock()
        self._history = []  # Audit trail
    
    def write(self, key: str, value, agent_name: str):
        with self._lock:
            self._store[key] = value
            self._history.append({"action": "write", "key": key, "agent": agent_name})
    
    def read(self, key: str, default=None):
        with self._lock:
            return self._store.get(key, default)
    
    def send_message(self, msg: AgentMessage):
        with self._lock:
            self._message_queue.append(msg)
            self._history.append({"action": "message", "from": msg.sender, "to": msg.recipient})
    
    def get_messages(self, recipient: str) -> list:
        with self._lock:
            msgs = [m for m in self._message_queue if m.recipient == recipient]
            self._message_queue = [m for m in self._message_queue if m.recipient != recipient]
            return msgs

# === Demo: Agents communicating via Blackboard ===
board = Blackboard()

# Research agent writes findings
board.write("research_findings", {"competitors": 5, "market_size": "$2.1B"}, "Alice")

# Research agent sends structured message to Code agent
board.send_message(AgentMessage(
    sender="Alice", recipient="Bob", msg_type="task_result",
    conversation_id="conv-001",
    payload={"findings": board.read("research_findings"), "recommendation": "proceed"},
    priority="high"
))

# Code agent reads messages
messages = board.get_messages("Bob")

print("📋 Blackboard Shared Memory Demo")
print("=" * 50)
print(f"  Stored data: {board.read('research_findings')}")
print(f"  Messages for Bob: {len(messages)}")
for msg in messages:
    print(f"    From: {msg.sender}, Type: {msg.msg_type}, Priority: {msg.priority}")
    print(f"    Payload: {json.dumps(msg.payload, indent=4)[:200]}")
print(f"\n  Audit trail: {len(board._history)} operations")
for h in board._history:
    print(f"    {h}")

---

## 3. Failure Handling & Circuit Breakers

Production multi-agent systems **will** fail. The question is: how gracefully?

| Failure Mode | Impact | Mitigation |
|-------------|--------|------------|
| Agent timeout | One agent blocks the pipeline | Deadline per agent (e.g., 30s max) |
| LLM hallucination | Agent produces garbage output | Output validation via Pydantic schemas |
| Infinite delegation loop | Agent A asks B, B asks A, repeat | Max delegation depth + cycle detection |
| Budget exhaustion | Runaway API costs | Hard budget cap per agent AND per workflow |
| Cascading failure | One agent fails, all downstream fail | Circuit breaker pattern |

In [ ]:
# Cell 5 — Circuit Breaker for multi-agent systems

class CircuitBreaker:
    """Prevents cascading failures in multi-agent pipelines.
    
    States: CLOSED (normal) → OPEN (failing, reject all) → HALF_OPEN (testing recovery)
    
    When an agent fails N times consecutively, the circuit OPENS
    and all requests to that agent are immediately rejected.
    After a cooldown period, it moves to HALF_OPEN to test recovery.
    """
    def __init__(self, failure_threshold: int = 3, cooldown_seconds: float = 30.0):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown_seconds
        self.state = "CLOSED"  # CLOSED, OPEN, HALF_OPEN
        self.failure_count = 0
        self.last_failure_time = 0
    
    def can_execute(self) -> bool:
        if self.state == "CLOSED":
            return True
        if self.state == "OPEN":
            if time.time() - self.last_failure_time > self.cooldown:
                self.state = "HALF_OPEN"
                return True  # Allow one probe request
            return False
        return True  # HALF_OPEN: allow probe
    
    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"
    
    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"

# Demo: Circuit breaker in action
breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)

print("⚡ Circuit Breaker Demo")
print("=" * 50)
for i in range(6):
    can_run = breaker.can_execute()
    if can_run:
        # Simulate failure for first 4 attempts
        if i < 4:
            breaker.record_failure()
            print(f"  Attempt {i+1}: FAILED (failures={breaker.failure_count}, state={breaker.state})")
        else:
            breaker.record_success()
            print(f"  Attempt {i+1}: SUCCESS (state={breaker.state})")
    else:
        print(f"  Attempt {i+1}: REJECTED by circuit breaker (state={breaker.state})")

print(f"\n📌 After 3 failures, the breaker OPENS and rejects all requests.")
print(f"   This prevents cascading failures from overwhelming the system.")

---

## 4. A2A Protocols: The HTTP of Agents

### 4.1 Google's A2A Specification

In April 2025, Google published the **Agent-to-Agent (A2A) protocol** — an open standard for agents to discover, authenticate, and communicate across organizational boundaries.

Think of A2A as **HTTP for AI agents**: just as any web server can talk to any browser via HTTP, any A2A-compliant agent can talk to any other.

**Core A2A concepts:**

| Concept | HTTP Analogy | What It Does |
|---------|-------------|-------------|
| **Agent Card** | `/robots.txt` | Declares agent capabilities, auth requirements |
| **Task** | HTTP Request | A unit of work sent to a remote agent |
| **Artifact** | HTTP Response body | Files/data returned by the agent |
| **Message** | HTTP headers | Metadata and conversation context |

### 4.2 Cross-Organization Agent Communication

In [ ]:
# Cell 6 — A2A Protocol: Agent Card + Cross-org communication

class AgentCard(BaseModel):
    """Agent Card — the A2A equivalent of a service discovery document.
    
    Published at: https://your-company.com/.well-known/agent.json
    Any agent can read this to discover capabilities.
    """
    name: str
    description: str
    url: str  # Agent endpoint
    version: str = "1.0"
    capabilities: List[str] = []
    authentication: dict = {}  # {"type": "bearer", "token_url": "..."}
    input_schema: dict = {}    # JSON Schema for accepted inputs
    output_schema: dict = {}   # JSON Schema for outputs
    rate_limit: str = "100/min"

class A2ATask(BaseModel):
    """A task sent from one agent to another via A2A protocol."""
    task_id: str
    sender_agent: str
    recipient_agent: str
    intent: str
    payload: dict
    auth_token: str
    conversation_id: str
    timeout_seconds: int = 30

class A2AResponse(BaseModel):
    """Response from a remote agent."""
    task_id: str
    status: Literal["completed", "failed", "pending", "requires_input"]
    result: dict
    artifacts: List[dict] = []
    cost_usd: float = 0.0

# === Simulate cross-org A2A communication ===

# Company A: Travel booking agent
travel_card = AgentCard(
    name="TravelBooker-v2",
    description="Books flights, hotels, and rental cars",
    url="https://travelco.com/api/agent",
    capabilities=["flight_search", "hotel_booking", "car_rental"],
    authentication={"type": "oauth2", "token_url": "https://travelco.com/oauth/token"},
    input_schema={"type": "object", "properties": {"destination": {"type": "string"}, "dates": {"type": "object"}}},
    rate_limit="50/min",
)

# Company B: Expense reporting agent
expense_card = AgentCard(
    name="ExpenseBot-v1",
    description="Processes expense reports and enforces travel policy",
    url="https://corpco.com/api/expense-agent",
    capabilities=["receipt_processing", "policy_check", "approval_routing"],
    authentication={"type": "bearer"},
)

# Step 1: Discovery — read the Agent Card
print("🤝 A2A Cross-Organization Communication")
print("=" * 60)
print(f"\n📇 Agent Card (Company A — Travel):")
print(f"   Name: {travel_card.name}")
print(f"   URL:  {travel_card.url}")
print(f"   Capabilities: {travel_card.capabilities}")
print(f"   Auth: {travel_card.authentication['type']}")
print(f"   Rate limit: {travel_card.rate_limit}")

# Step 2: Send A2A task
task = A2ATask(
    task_id=str(uuid.uuid4())[:8],
    sender_agent="corpco/expense-bot",
    recipient_agent="travelco/travel-booker",
    intent="flight_search",
    payload={"destination": "NYC", "departure": "2026-05-15", "return": "2026-05-18", "budget_max": 500},
    auth_token="Bearer eyJhbG...",
    conversation_id="conv-" + str(uuid.uuid4())[:8],
)

print(f"\n📤 A2A Task Sent:")
print(json.dumps(task.model_dump(), indent=2))

# Step 3: Receive response
response = A2AResponse(
    task_id=task.task_id,
    status="completed",
    result={"flights": [{"airline": "Delta", "price": 342, "departure": "08:15"}, {"airline": "United", "price": 389, "departure": "10:30"}]},
    artifacts=[{"type": "itinerary_pdf", "url": "https://travelco.com/itineraries/abc123.pdf"}],
    cost_usd=0.003,
)

print(f"\n📥 A2A Response Received:")
print(json.dumps(response.model_dump(), indent=2))

print(f"\n📌 Key insight: Travel agent and Expense agent are from DIFFERENT companies.")
print(f"   A2A protocol lets them communicate securely via standardized schemas.")
print(f"   This is how enterprise AI will work — agents negotiating B2B transactions.")

---

## 5. Framework Comparison (2026)

| Framework | Paradigm | Multi-Agent? | Best For | Trade-off |
|-----------|----------|:----------:|----------|----------|
| **LangGraph** | State machine graph | ✅ | Production, deterministic workflows | More boilerplate, maximum control |
| **CrewAI** | Role-based teams | ✅ | Fast prototyping, hierarchical teams | Less control, higher-level |
| **AutoGen** | Conversational agents | ✅ | Research, open-ended problems | Risk of infinite loops |
| **Swarm** (OpenAI) | Lightweight handoffs | ✅ | Simple routing between agents | Stateless, limited memory |
| **Google ADK** | A2A-native toolkit | ✅ | Cross-org agent systems | Newer, evolving |

### ⚖️ The Fundamental Trade-off

```
High Autonomy (AutoGen)                    High Determinism (LangGraph)
├── Agents self-organize                   ├── You define the exact graph
├── Can solve unexpected problems           ├── Solves specific problems reliably
├── Risk: infinite loops, off-topic         ├── Safe, predictable execution
└── Best: Research, exploration            └── Best: Production enterprise
```

**Production recommendation:** Start with **LangGraph** for control. Use **CrewAI** for rapid prototyping. Add **A2A** when you need cross-organization agent communication.

---

## ✅ Knowledge Check

### Q1: Why is the Manager-Worker pattern more reliable than peer-to-peer agents?
<details><summary>👉 Answer</summary>

Manager-Worker has a single point of control that enforces execution order, budget limits, and failure handling. Peer-to-peer agents can deadlock (A waits for B, B waits for A), overspend (no central budget tracker), or loop infinitely (no one enforces max steps). The Manager acts as a deterministic orchestrator — exactly the pattern from NB24.
</details>

### Q2: Why must agents communicate via Pydantic schemas, not free-form text?
<details><summary>👉 Answer</summary>

Free-form text is unparseable and non-deterministic. Agent A might say "I found 3 results" in one format and "three items discovered" in another — Agent B can't reliably extract the number. Pydantic schemas enforce a contract: the sender MUST produce valid JSON matching the schema, and the receiver can parse it deterministically. This is the same principle as API contracts in microservices.
</details>

### Q3: An agent fails 5 times in a row. What should happen?
<details><summary>👉 Answer</summary>

The **circuit breaker** should open, immediately rejecting all further requests to that agent. This prevents: (1) wasting API budget on a broken agent, (2) cascading failures to downstream agents, (3) increased latency from repeated timeouts. After a cooldown period (e.g., 30s), the breaker moves to HALF_OPEN to test if the agent has recovered.
</details>

### Q4: What is an Agent Card and why is it published at `.well-known/agent.json`?
<details><summary>👉 Answer</summary>

An Agent Card is a machine-readable document that declares an agent's capabilities, authentication requirements, input/output schemas, and rate limits. Publishing it at `.well-known/agent.json` follows the same convention as `.well-known/openid-configuration` — it's a standardized discovery endpoint. Any agent can read another agent's card to learn how to communicate with it, enabling cross-organization interoperability without manual integration.
</details>

### Q5: When would you use CrewAI vs LangGraph for a multi-agent system?
<details><summary>👉 Answer</summary>

**CrewAI:** Rapid prototyping, hackathons, when you want to define agents by role/backstory and let the framework handle orchestration. Higher-level, less control. **LangGraph:** Production systems where you need exact control over the execution graph, conditional routing, checkpointing, and human-in-the-loop. More boilerplate, but fully deterministic and debuggable.
</details>

---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Add a `TestAgent` to the team that runs `pytest` on the code artifact and returns a pass/fail result with test coverage percentage.
2. Extend the `Blackboard` class with a `subscribe(key, callback)` method that triggers a callback whenever a key is updated (the Observer pattern).

### Tier 2: Intermediate
1. **Async Multi-Agent Pipeline:** Refactor the Manager to run Research and a parallel DataAgent simultaneously using `asyncio.gather()`, then pass both results to the CodeAgent.
2. **Agent Registry & Discovery:** Build an `AgentRegistry` class that stores Agent Cards and provides a `discover(capability)` method to find agents by capability.

### Tier 3: Advanced
1. **Full LangGraph Multi-Agent:** Install `langgraph` and build a 3-agent graph where the Manager node routes to Research, Code, or Review nodes via conditional edges. Add SQLite checkpointing.
2. **Cross-Org A2A Server:** Build a FastAPI server that exposes an Agent Card at `/.well-known/agent.json` and accepts A2A tasks at `/api/tasks`. Test with `httpx`.

---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Production Impact |
|---------|-----------------|------------------|
| Multi-agent teams | Decompose problems into specialist agents | Better quality via separation of concerns |
| Blackboard pattern | Shared memory for agent communication | Decoupled agents, audit trail |
| Circuit breakers | Prevent cascading failures | System resilience |
| A2A protocols | Standardized cross-org agent communication | Enterprise interoperability |
| Framework landscape | LangGraph (production) vs CrewAI (prototyping) | Right tool for the job |

**Next →** `32_model_context_protocol.ipynb` — The Model Context Protocol (MCP): Universal AI Tool Integration